# Lending Club: Which Borrower Characteristics Are Linked to Default?
### Project 2: Python EDA (pandas, matplotlib, seaborn)

**Business question:** Project 1 (SQL) showed that loan grade and loan term are strongly linked to default. This project looks at the **borrower profile** (income, debt, employment, home ownership, credit behaviour) and asks:

1. Which borrower characteristics are linked to higher default?
2. Do those patterns still hold **within the same loan grade**, or were they already captured by the grade?
3. Which characteristics separate low-risk from high-risk borrowers the most?

**Data:** `loan_clean.csv` = 100,000 completed Lending Club loans (Fully Paid or Charged Off) prepared in Project 1. `is_default = 1` means the loan was Charged Off.

**Reminder:** results show association, not causation. Active (Current) loans are not in this data.

## Step 1: Set up and load the data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

BLUE, ORANGE, GREY = "#2b6cb0", "#dd6b20", "#718096"
os.makedirs("images", exist_ok=True)

def save(fig, name):
    """Save a chart into the images folder and show it."""
    fig.tight_layout()
    fig.savefig(f"images/{name}.png", dpi=150)
    plt.show()

In [ ]:
# If loan_clean.csv is not in this Colab session, a file picker will open. Upload it.
if not os.path.exists("loan_clean.csv"):
    from google.colab import files
    print("Upload loan_clean.csv")
    files.upload()

df = pd.read_csv("loan_clean.csv")
print("Rows, columns:", df.shape)
df.head()

## Step 2: Check data quality
Before analysing, check for duplicates, missing values and unusual numbers.

In [ ]:
print("Duplicate loan_id:", df["loan_id"].duplicated().sum())

missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nColumns with missing values:")
print(pd.DataFrame({"missing": missing, "pct": (missing / len(df) * 100).round(2)}))

print("\nSummary of key numeric columns:")
print(df[["loan_amnt", "int_rate", "annual_inc", "dti", "revol_util"]].describe().round(2))

**How to read this:** income is usually right-skewed (a few very high earners), so medians and capped charts are more useful than averages. `dti` and `revol_util` can have odd values (negative or above 100), so the bands below handle them explicitly.

## Step 3: Create features
Raw numbers are hard to compare, so we turn them into **groups (bands)** and a numeric employment length.

In [ ]:
# Employment length: "10+ years" -> 10, "< 1 year" -> 0, "5 years" -> 5
def emp_to_years(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    if x.startswith("<"):
        return 0
    digits = "".join(ch for ch in x if ch.isdigit())
    return int(digits) if digits else np.nan

df["emp_years"] = df["emp_length"].apply(emp_to_years)

df["emp_group"] = pd.cut(df["emp_years"], bins=[-1, 0, 3, 6, 9, 10],
                         labels=["<1 yr", "1-3 yrs", "4-6 yrs", "7-9 yrs", "10+ yrs"])
df["emp_group"] = df["emp_group"].cat.add_categories("Unknown").fillna("Unknown")

# Income, debt and credit-usage bands
df["income_band"] = pd.cut(df["annual_inc"], bins=[0, 30000, 50000, 75000, 100000, 150000, np.inf],
                           labels=["<30k", "30-50k", "50-75k", "75-100k", "100-150k", "150k+"], right=False)
df["dti_band"] = pd.cut(df["dti"], bins=[0, 10, 20, 30, 40, np.inf],
                        labels=["0-10", "10-20", "20-30", "30-40", "40+"], right=False)
df["util_band"] = pd.cut(df["revol_util"], bins=[0, 25, 50, 75, 100, np.inf],
                         labels=["0-25%", "25-50%", "50-75%", "75-100%", "100%+"], right=False)

# Loan size relative to income
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"]
df["lti_band"] = pd.cut(df["loan_to_income"], bins=[0, 0.1, 0.2, 0.3, np.inf],
                        labels=["<10%", "10-20%", "20-30%", "30%+"], right=False)

# Past credit problems
df["delinq_group"] = pd.cut(df["delinq_2yrs"], bins=[-0.5, 0.5, 1.5, 2.5, np.inf],
                            labels=["0", "1", "2", "3+"])
df["pubrec_group"] = pd.cut(df["pub_rec"], bins=[-0.5, 0.5, 1.5, np.inf],
                            labels=["0", "1", "2+"])

df[["emp_length", "emp_group", "annual_inc", "income_band", "dti", "dti_band",
    "revol_util", "util_band", "lti_band"]].head(8)

## Step 4: Look at the distributions
What do typical loans and borrowers look like?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0, 0].hist(df["loan_amnt"], bins=30, color=BLUE)
axes[0, 0].set_title("Loan amount")

axes[0, 1].hist(df["int_rate"], bins=30, color=BLUE)
axes[0, 1].set_title("Interest rate (%)")

cap = df["annual_inc"].quantile(0.99)
axes[1, 0].hist(df.loc[df["annual_inc"] <= cap, "annual_inc"], bins=30, color=BLUE)
axes[1, 0].set_title(f"Annual income (up to 99th percentile, {cap:,.0f})")

axes[1, 1].hist(df.loc[df["dti"].between(0, 60), "dti"], bins=30, color=BLUE)
axes[1, 1].set_title("Debt-to-income ratio (0 to 60)")

save(fig, "01_distributions")

In [ ]:
grades = sorted(df["grade"].dropna().unique())

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x="grade", y="int_rate", order=grades, color=BLUE, ax=ax)
ax.set_title("Interest rate by loan grade")
ax.set_xlabel("Loan grade (A = best, G = worst)")
ax.set_ylabel("Interest rate (%)")
sns.despine(ax=ax)
save(fig, "02_interest_by_grade")

**What to look for:** the shape of each distribution (skewed or balanced), and whether interest rates step up from grade to grade (the box for each grade should sit higher than the one before).

## Step 5: Default rate by borrower characteristic
The helper below does the same three things for every characteristic: **group the loans, calculate the default rate, draw a bar chart**. The group size (`n`) is shown under each bar, and groups with fewer than 500 loans are left out because their rates are not reliable.

In [ ]:
OVERALL = df["is_default"].mean() * 100
print(f"Overall default rate: {OVERALL:.2f}%")

tables = {}     # saves every result so we can rank the drivers later
chart_no = 2    # chart file numbering continues from the charts above

def default_table(col, min_loans=500):
    """Loans and default rate for each group of a column."""
    t = (df.groupby(col, observed=True)["is_default"]
           .agg(loans="size", default_pct="mean")
           .reset_index())
    t["default_pct"] = (t["default_pct"] * 100).round(2)
    t = t[t["loans"] >= min_loans]
    if isinstance(df[col].dtype, pd.CategoricalDtype):
        t = t.sort_values(col)                                   # keep natural band order
    else:
        t = t.sort_values("default_pct", ascending=False)        # highest risk first
    return t.reset_index(drop=True)

def driver(name, col, min_loans=500):
    """Bar chart of default rate by group, with the overall rate as a dashed line."""
    global chart_no
    t = default_table(col, min_loans)
    t["group"] = t[col].astype(str)
    tables[name] = t[["group", "loans", "default_pct"]]

    labels = [f"{g}\n(n={n:,})" for g, n in zip(t["group"], t["loans"])]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(labels, t["default_pct"], color=BLUE)
    ax.axhline(OVERALL, color=GREY, linestyle="--", linewidth=1, label=f"Overall default rate {OVERALL:.1f}%")
    for b, v in zip(bars, t["default_pct"]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)
    ax.set_title(f"Default rate by {name}")
    ax.set_ylabel("Default rate (%)")
    ax.set_ylim(0, max(t["default_pct"].max(), OVERALL) * 1.35)
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    if len(t) > 6:
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    sns.despine(ax=ax)
    chart_no += 1
    save(fig, f"{chart_no:02d}_{name.lower().replace(' ', '_')}")
    return tables[name]

### 5.1 Home ownership
Do renters, mortgage holders and owners default at different rates?

In [ ]:
driver("home ownership", "home_ownership")

### 5.2 Income verification
Does it matter whether Lending Club verified the borrower's income?

In [ ]:
driver("income verification", "verification_status")

### 5.3 Employment length
Do borrowers with longer employment history default less?

In [ ]:
driver("employment length", "emp_group")

### 5.4 Annual income
Do higher-income borrowers default less?

In [ ]:
driver("annual income", "income_band")

### 5.5 Debt-to-income ratio (DTI)
DTI = monthly debt payments divided by monthly income. Does a heavier debt load mean more default?

In [ ]:
driver("debt-to-income", "dti_band")

### 5.6 Credit card utilization
`revol_util` is the share of available revolving credit the borrower is using.

In [ ]:
driver("credit utilization", "util_band")

### 5.7 Loan size relative to income
A $20,000 loan means something different for a $40k earner and a $150k earner.

In [ ]:
driver("loan-to-income", "lti_band")

### 5.8 Recent delinquencies (last 2 years)
Does a recent history of late payments predict default?

In [ ]:
driver("delinquencies", "delinq_group")

### 5.9 Public records
Public records include things like bankruptcies.

In [ ]:
driver("public records", "pubrec_group")

### 5.10 Loan purpose
Project 1 looked at purpose with SQL. Here we repeat it in Python as a cross-check (only purposes with at least 500 loans).

In [ ]:
driver("loan purpose", "purpose")

**What to look for in all of Step 5:** whether the bars climb or fall in a steady way across the groups (a clear pattern) or look flat and close to the dashed overall line (a weak driver). Small groups (`n` under the chart title area) deserve less trust.

## Step 6: Correlations
Correlation measures how two numbers move together (from -1 to +1). It only captures straight-line relationships, so treat it as a quick screen, not a verdict.

Columns such as `total_pymnt` and `recoveries` are left out on purpose: they are only known **after** a loan ends, so they would trivially "predict" default.

In [ ]:
num_cols = ["loan_amnt", "int_rate", "annual_inc", "dti", "delinq_2yrs", "revol_bal",
            "revol_util", "open_acc", "total_acc", "pub_rec", "emp_years", "is_default"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, annot_kws={"size": 8})
ax.grid(False)
plt.setp(ax.get_yticklabels(), rotation=0)
ax.set_title("Correlation between numeric columns")
chart_no += 1
save(fig, f"{chart_no:02d}_correlation_heatmap")

print("Correlation with is_default (strongest first):")
print(corr["is_default"].drop("is_default").sort_values(key=abs, ascending=False).round(3))

## Step 7: Does the pattern hold within the same loan grade?
Loan grade already reflects Lending Club's own risk assessment. If a pattern **disappears inside each grade**, it was probably already priced into the grade. This is the same idea as Project 1, where loan size turned out to be explained by loan term.

Cells with fewer than 200 loans are left blank.

In [ ]:
def grade_heatmap(band_col, label, min_loans=200):
    """Default rate for every grade x band combination."""
    global chart_no
    counts = df.pivot_table(index="grade", columns=band_col, values="is_default",
                            aggfunc="count", observed=True)
    rates = df.pivot_table(index="grade", columns=band_col, values="is_default",
                           aggfunc="mean", observed=True) * 100
    rates = rates.where(counts >= min_loans)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(rates, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
                cbar_kws={"label": "Default rate (%)"})
    ax.grid(False)
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.set_title(f"Default rate by loan grade and {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Loan grade")
    chart_no += 1
    save(fig, f"{chart_no:02d}_grade_by_{label.lower().replace(' ', '_').replace('-', '_')}")
    return rates.round(1)

grade_heatmap("dti_band", "debt-to-income")

In [ ]:
grade_heatmap("util_band", "credit utilization")

In [ ]:
grade_heatmap("income_band", "annual income")

**How to read the heatmaps:** go down a column to see the effect of grade, go across a row to see the effect of the borrower characteristic **with grade held constant**. If the colour changes across a row, the characteristic adds information beyond the grade. If each row is roughly one colour, it does not.

## Step 8: Rank the drivers
For each characteristic, compare the **lowest-risk group** with the **highest-risk group**. The bigger the gap (in percentage points), the more that characteristic separates safe from risky borrowers.

In [ ]:
rows = []
for name, t in tables.items():
    lo = t.loc[t["default_pct"].idxmin()]
    hi = t.loc[t["default_pct"].idxmax()]
    rows.append({
        "driver": name,
        "lowest-risk group": lo["group"],
        "lowest default %": lo["default_pct"],
        "highest-risk group": hi["group"],
        "highest default %": hi["default_pct"],
        "spread (pts)": round(hi["default_pct"] - lo["default_pct"], 2),
    })

summary = pd.DataFrame(rows).sort_values("spread (pts)", ascending=False).reset_index(drop=True)
summary.to_csv("driver_summary.csv", index=False)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
s = summary.sort_values("spread (pts)")
ax.barh(s["driver"], s["spread (pts)"], color=BLUE)
for y, v in enumerate(s["spread (pts)"]):
    ax.text(v + 0.2, y, f"{v:.1f}", va="center", fontsize=9)
ax.set_title("Gap in default rate between highest- and lowest-risk group")
ax.set_xlabel("Percentage points")
sns.despine(ax=ax)
chart_no += 1
save(fig, f"{chart_no:02d}_driver_ranking")

**Careful:** the spread depends on how the groups were cut and on group sizes, so it is a rough ranking, not a precise measure of importance.

## Step 9: Print all results (to copy into your README)
Run this cell and copy the whole output. It contains every table you need to write the findings.

In [ ]:
print(f"Overall default rate: {OVERALL:.2f}%   (rows: {len(df):,})\n")
for name, t in tables.items():
    print(f"== {name} ==")
    print(t.to_string(index=False))
    print()
print("== ranking of drivers ==")
print(summary.to_string(index=False))

## Step 10: Write your findings
Use this template in your README (replace the brackets with your numbers):

- **Overall:** [X]% of the [N] loans defaulted.
- **Strongest borrower-level driver:** [driver] separates [lowest group] ([x]%) from [highest group] ([y]%), a gap of [z] points.
- **Weak or flat drivers:** [driver] shows little difference between groups.
- **Within grade:** [driver] still matters inside each grade / mostly disappears once grade is held constant.
- **Caution:** association only; Current loans are excluded; groups under 500 loans were removed.

## Step 11: Download your charts and summary (Colab only)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("project2_images", "zip", "images")
files.download("project2_images.zip")
files.download("driver_summary.csv")